In [1]:
import os
import random
import shutil
from ultralytics import YOLO
from PIL import Image


ModuleNotFoundError: No module named 'ultralytics'

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!cp -r /content/drive/MyDrive/DeepLearningProject/data/train/task1train540p /content/

In [7]:
TASK1_PATH = "/content/drive/MyDrive/DeepLearningProject/data/train/task1train540p"

classes = set()

for file in os.listdir(TASK1_PATH):
    if file.endswith(".txt"):
        with open(os.path.join(TASK1_PATH, file), "r") as f:
            line = f.readline().strip()
            class_name = line.split(":")[0]
            classes.add(class_name)

print("Total Classes:", len(classes))
print("Class Names:", classes)

Total Classes: 4
Class Names: {'left-hand-curve', 'right-hand-curve', 'side-road-left', 'gap-in-median'}


In [8]:
sample_img = "664_frame_1144.png"

img = Image.open(os.path.join(TASK1_PATH, sample_img))
print("Image Size:", img.size)

Image Size: (960, 540)


In [9]:
class_names = [
    'gap-in-median',
    'right-hand-curve',
    'side-road-left',
    'left-hand-curve'
]

class_to_id = {name: idx for idx, name in enumerate(class_names)}

print(class_to_id)

{'gap-in-median': 0, 'right-hand-curve': 1, 'side-road-left': 2, 'left-hand-curve': 3}


In [10]:
YOLO_BASE = "/content/drive/MyDrive/DeepLearningProject/dataset_yolo"

folders = [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]

for folder in folders:
    os.makedirs(os.path.join(YOLO_BASE, folder), exist_ok=True)

print("YOLO folder structure created successfully!")

YOLO folder structure created successfully!


In [11]:
all_images = [f for f in os.listdir(TASK1_PATH) if f.endswith(".png")]

random.shuffle(all_images)

split_index = int(0.8 * len(all_images))

train_images = all_images[:split_index]
val_images = all_images[split_index:]

print("Total Images:", len(all_images))
print("Train Images:", len(train_images))
print("Validation Images:", len(val_images))

Total Images: 2000
Train Images: 1600
Validation Images: 400


# Phase 3

In [12]:
IMAGE_WIDTH = 960
IMAGE_HEIGHT = 540

def convert_to_yolo(image_list, split_type):

    for image_file in image_list:

        base_name = image_file.replace(".png", "")
        label_file = base_name + ".txt"

        src_img = os.path.join(TASK1_PATH, image_file)
        dst_img = os.path.join(YOLO_BASE, f"images/{split_type}", image_file)
        shutil.copy(src_img, dst_img)

        with open(os.path.join(TASK1_PATH, label_file), "r") as f:
            line = f.readline().strip()

        class_name, coords = line.split(":")
        x, y, w, h = map(int, coords.split(","))

        class_id = class_to_id[class_name]

        center_x = (x + w/2) / IMAGE_WIDTH
        center_y = (y + h/2) / IMAGE_HEIGHT
        norm_w = w / IMAGE_WIDTH
        norm_h = h / IMAGE_HEIGHT

        yolo_label_path = os.path.join(YOLO_BASE, f"labels/{split_type}", base_name + ".txt")

        with open(yolo_label_path, "w") as f:
            f.write(f"{class_id} {center_x} {center_y} {norm_w} {norm_h}\n")

convert_to_yolo(train_images, "train")
convert_to_yolo(val_images, "val")

print("Conversion Complete")

Conversion Complete


In [13]:
print(len(os.listdir(os.path.join(YOLO_BASE, "images/train"))))
print(len(os.listdir(os.path.join(YOLO_BASE, "labels/train"))))
print(len(os.listdir(os.path.join(YOLO_BASE, "images/val"))))
print(len(os.listdir(os.path.join(YOLO_BASE, "labels/val"))))

1600
1600
400
400


In [14]:
import yaml

data_yaml = {
    'train': '/content/drive/MyDrive/DeepLearningProject/dataset_yolo/images/train',
    'val': '/content/drive/MyDrive/DeepLearningProject/dataset_yolo/images/val',
    'nc': 4,
    'names': [
        'gap-in-median',
        'right-hand-curve',
        'side-road-left',
        'left-hand-curve'
    ]
}

yaml_path = "/content/drive/MyDrive/DeepLearningProject/dataset_yolo/data.yaml"

with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)

print("data.yaml created successfully!")

data.yaml created successfully!


In [16]:
model = YOLO("yolov8n.pt")

model.train(
    data="/content/drive/MyDrive/DeepLearningProject/dataset_yolo/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    name="traffic_sign_detector"
)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (AMD EPYC 7B12)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/DeepLearningProject/dataset_yolo/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=traffic_sign_detector2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ov

KeyboardInterrupt: 